# ScoutCard AI: a game's film → scout cards, on a GPU

Runtime → Change runtime type → **T4 GPU** first. Then run the cells top to bottom (one at a time is easiest to follow).

On your Mac, put the game's clips (and each clip's calibration `.json` if you made them by hand) in
`video-service/clips/`, and the breakdown sheets in `video-service/` (`hudl.csv`, and the coach's corrected
`corrected.csv` if you have it). Then make the one file this notebook takes, the whole project:

```bash
cd ~ && rm -f ~/Desktop/scoutai.zip
cd ~/scoutai && git pull origin claude/scoutcard-ai-v1-scaffold-c4bgpf && cd ~
zip -r ~/Desktop/scoutai.zip scoutai -x "scoutai/node_modules/*" "scoutai/.next/*" "scoutai/.git/*" \
  "scoutai/video-service/.venv/*" "*/__pycache__/*" "scoutai/video-service/recordings/*" \
  "scoutai/video-service/tracks/*" "scoutai/video-service/detections/*" "scoutai/video-service/debug/*" "scoutai/video-service/game-work/*"
```

The clips are game film (minors): use your own Google account, and delete the runtime when you're done
(Runtime → Disconnect and delete runtime).

In [ ]:
!nvidia-smi -L
import torch
print("cuda:", torch.cuda.is_available())

## 1. Upload `clips.zip` (just the three recordings: play_08.mp4, play_34.mp4, play_43.mp4). The code, calibration files and sheets come from GitHub. (A whole-project `scoutai.zip` still works.)

In [ ]:
import os, glob
os.chdir("/content")
BRANCH = "claude/scoutcard-ai-v1-scaffold-c4bgpf"
if os.path.exists("scoutai.zip"):
    # The whole project zip (code + clips), as before.
    !rm -rf scoutai && unzip -q scoutai.zip
else:
    # Just the recordings: drag them (and each clip's calibration .json) into one `clips.zip`, upload it here, and the code comes from GitHub.
    if not os.path.exists("clips.zip"):
        from google.colab import files
        files.upload()  # pick clips.zip
    !rm -rf scoutai scoutai-code && mkdir scoutai-code && curl -fsSL https://codeload.github.com/jpt-16/scoutai/tar.gz/refs/heads/{BRANCH} | tar xz -C scoutai-code --strip-components=1 && mv scoutai-code scoutai
    !mkdir -p scoutai/video-service/clips && unzip -qjo clips.zip -d scoutai/video-service/clips
    # The three plays' calibration files and sheets ship with the code: use them unless the zip brought its own.
    T = "scoutai/video-service/eval/truth/plays_8_34_43"
    !cp -n {T}/play_*.json scoutai/video-service/clips/ 2>/dev/null; cp -n {T}/hudl.csv {T}/corrected.csv scoutai/video-service/ 2>/dev/null; true
    for f in glob.glob("scoutai/video-service/clips/*.csv"):
        !mv "{f}" scoutai/video-service/
!ls scoutai/video-service/clips | head -20 && ls scoutai/video-service/*.csv 2>/dev/null

## 2. Install (Python for the film, Node for the cards and the score; about two minutes)

In [ ]:
!pip install -q "opencv-python-headless==5.0.0.93" transformers timm pillow pytest
!node -e "process.exit(+process.versions.node.split('.')[0] < 20 ? 1 : 0)" || (curl -fsSL https://deb.nodesource.com/setup_20.x | bash - >/dev/null && apt-get install -y -qq nodejs >/dev/null)
!node --version
%cd /content/scoutai
!npm install --no-audit --no-fund --loglevel=error

## 3. The whole game
Calibrates each clip (a hand `.json` beside a clip wins), reads the formation, follows every player, reads routes, runs and the defense, and writes `game.json`. With a corrected sheet it also prints the score. Change the file names here if yours differ.

In [ ]:
CHECKPOINT = "PekingU/rtdetr_v2_r50vd"
CONFIDENCE = "0.3"
SHEET = "hudl.csv"        # Hudl's breakdown for this game ("" if none)
KEY = "corrected.csv"     # the coach's corrected breakdown, to score against ("" if none)
SCOUT_COLOR = "red"       # the jersey color of the team being scouted ("" if you want both teams' cards)
TRACKER = "boxes"         # "boxes" (each man by his own box; the default, best on the real plays), "greedy" or "tracklets"
SECONDS = 12              # how long to follow after the calibrated frame (a recording has 8-10 s of set before the snap)
import os
os.environ["MODEL_CHECKPOINT"] = CHECKPOINT
os.environ["DETECT_CONFIDENCE"] = CONFIDENCE
os.environ["TRACKER"] = TRACKER
%cd /content/scoutai/video-service
args = f"clips/ --out game.json --work game-work --detections --seconds {SECONDS}"
if SCOUT_COLOR: args += f" --scout-color {SCOUT_COLOR}"
if SHEET and os.path.exists(SHEET): args += f" --sheet {SHEET}"
if KEY and os.path.exists(KEY): args += f" --key {KEY}"
print("python -m eval.game", args)
!python -m eval.game {args} 2>&1 | tee game-work-log.txt

## 4. Download the results
`game.json` opens on the app's landing page (Open a game file). Send `results.zip` back too: it has the score, the log and the detections (numbers, no film).

In [ ]:
%cd /content/scoutai/video-service
!zip -qr results.zip game.json game-work-log.txt game-work/tracks game-work/detections game-work/ours.csv 2>/dev/null
from google.colab import files
files.download("game.json")
files.download("results.zip")